# AssistAI
This is assist AI a locally run LLM that has access to your files so it knows who you are

Sources:
- https://github.com/techleadhd/chatgpt-retrieval/blob/main/chatgpt.py
- https://python.langchain.com

## First expirement

I am try and make a call to the locally run model and see if it answers, ideally I want to also give it some context and see if it can answer my questions

In [2]:
import os
import sys

# The overall model
from langchain_community.document_loaders import TextLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

In [32]:
# Test if llm is loaded
# llm.invoke("how can langsmith help with testing?")

In [33]:
# Ask question based on one document
loader = TextLoader(os.path.join("data", "dogs.md"))
document = loader.load()

prompt = ChatPromptTemplate.from_messages([
    ("system", "Pretend like the only knowledge you have is from the context given"),
    ("user", "{input}")
])

output_parser = StrOutputParser()

chain = prompt | llm | output_parser

chain.invoke({
    "input": "what is the difference between a dog penis and a human penis?",
    "context": document
})

"The human penis is a long, slender penis that grows outside of a man's body. It is typically around 3 to 4 inches (7.6-10 cm) in length, with a diameter ranging from 1.5 to 2 inches (3.8-5.1 cm). On the other hand, the dog penis is a small and short penis that grows inside of a man's body. It typically measures around 1.5 to 2 inches (3.8-5.1 cm) in length. In general, dogs have shorter and more elongated penises than men due to their reproductive system being primarily focused on the female species."

### Conclusions from first experiment

In the first experiment I have found out that one document is not enough to give as context and this is not the correct way. However I have learnt how to use langchain to ask a question to my locally run model and to give it a single document as context

## Second experiment

In the first experiment I saw that it can answer my questions but that one document is by far not enough and that it still uses mostly it's own context. I want to try and give it all files in a folder as context and see how this changes things.

In [3]:
# The overall model
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

# Imports to load files
from langchain_community.document_loaders import DirectoryLoader
from langchain.chains.combine_documents import create_stuff_documents_chain

In [3]:
# Loading all documents
loader = DirectoryLoader('data', loader_cls=TextLoader, glob='**/*.md')
docs = loader.load()

print(len(docs))

222


In [4]:
# Create chain
prompt = ChatPromptTemplate.from_messages([
    ("system", "Find all answers in the documents provided as context \"{context}\""),
    ("user", "{input}")
])

combine_documents_chain = create_stuff_documents_chain(llm, prompt)

In [43]:
# Invoke chat
combine_documents_chain.invoke({
    "input": input("What is your question?"),
    "context": docs
})

'Ubuint is a free, open-source, and cross-platform software suite created by the University of California, Berkeley. It offers various tools for text editing, multimedia creation, scientific computing, and machine learning, among other features. Ubuint is known for its ease of use and versatility, making it an excellent choice for those looking to create innovative projects without sacrificing productivity or performance. With over 3000 contributors and a user base of more than 6 million, Ubuint has become one of the most popular open-source software suites available.'

### Conclusions from second experiment

This already works a lot better sicne there is a lot of context given, however the drawback currently is that my machine takes a long time to answer the question. This would not happen on a machine with a graphics card but this application should be usable for all even without having a GPU.

## Third experiment

In the last experiment there were two problems, the model didnt use the documents as it's main source so hallucinated and it took too long to generate a response.
In the third experiment I want to try and use a different chain called the `RetrievalChain` which first fetches all the corresponding documents before feeding them to the model.

In [4]:
import os

# The overall model
from langchain_community.document_loaders import TextLoader
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

# Imports to load files
from langchain_community.document_loaders import DirectoryLoader
from langchain.chains.combine_documents import create_stuff_documents_chain

# Imports to use vector store
from langchain_community.vectorstores import Chroma
from langchain.chains import create_retrieval_chain
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain import hub

In [9]:
# Loading all documents
loader = DirectoryLoader('data', loader_cls=TextLoader, glob='**/*.md')
docs = loader.load()

# Create chain
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_documents_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)
# Create new chain
embedding_function = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

PERSIST = False

if os.path.exists("persist") and PERSIST:
    vectorstore = Chroma.from_documents(docs, embedding_function, persist_directory="persist")
elif not os.path.exists("persist") and PERSIST:
    os.makedirs("persist")
    vectorstore = Chroma.from_documents(docs, embedding_function, persist_directory="persist")
else:
    vectorstore = Chroma.from_documents(docs, embedding_function)

retriever = vectorstore.as_retriever(k=10)

retrieval_chain = create_retrieval_chain(retriever, combine_documents_chain)

In [10]:
retrieval_chain.invoke({
  "input": "What frontend framework is best to create an application quickly"
})

{'input': 'What frontend framework is best to create an application quickly',
 'context': [Document(page_content="---\ntitle: Frameworks\nuuid: 4a2b2a1c-da0b-11ee-9a0e-ca8ae82b63ae\nversion: 3\ncreated: '2024-03-04T09:38:32Z'\ntags:\n  - javascript\n  - imported/markdown\n  - programming\n  - backend\n---\n\n# Frameworks\n\nFor Backend frameworks the same applies as frontend frameworks. There are almost too many flavours to choose from. I have to say that within this space I have only tried three options which are `Express.js`, `Nest.js` and `Koa`.\n\nLike in the Frontend frameworks I will go more in depth what dependencies I like to use with each framework and much more but here are some summarisations of what I like about each.\n\n### [Express.js](Express%20js.md)\n\nExpress prides itself about being a very minimalist framework and this shows. You can basically get a webserver working within **5 minutes** which is amazing. Like everything within the programming space this is a streng

### Conclusions from third experiment

In my third experiment I have tried to get all the documents into the model. However in my computer without a GPU this has caused the model to need a lot of time to respond. Some optimisation is needed.

## Fourth experiment

Finally I will try to optimise the model so it can hopefully run on computers with mid range specifications and get a response within 10 seconds.

In [7]:
import sys

# The overall model
from langchain_community.document_loaders import TextLoader
from langchain_community.llms import Ollama

llm = Ollama(model="tinyllama")

# Imports to load files
from langchain_community.document_loaders import DirectoryLoader
from langchain.chains.combine_documents import create_stuff_documents_chain

# Imports to use vector store
from langchain_community.vectorstores import Redis
from langchain.chains import create_retrieval_chain
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings,
)
from langchain import hub

In [8]:
redis_url = "redis://localhost:6379"

# Loading all documents
loader = DirectoryLoader('data', loader_cls=TextLoader, glob='**/*.md')
documents = loader.load()

# Create chain
retrieval_qa_chat_prompt = hub.pull("langchain-ai/retrieval-qa-chat")

combine_documents_chain = create_stuff_documents_chain(llm, retrieval_qa_chat_prompt)

# Split markdown documents
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
    ("###", "Header 3"),
]

# Create new chain
embedding = SentenceTransformerEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Redis.from_documents(documents=documents, embedding=embedding, redis_url=redis_url)

retriever = vectorstore.as_retriever(k=10)

retrieval_chain = create_retrieval_chain(retriever, combine_documents_chain)

In [31]:
retrieval_chain.invoke({
  "input": "What frontend framework is best to create an application quickly"
})

score_threshold is deprecated. Use distance_threshold instead.score_threshold should only be used in similarity_search_with_relevance_scores.score_threshold will be removed in a future release.


### Result of Third experiment



## Final test

In [ ]:
chat_history = []
query = None

while True:
  if not query:
    query = input("What do you want to know?: ")
  
  if query in ['quit', 'q', 'exit']:
    sys.exit()
  
  result = retrieval_chain.invoke({
      "input": query
  })
  print(result.get('answer', 'I do not know'))

  chat_history.append((query, result['answer']))
  query = None